# KLDM Notebook

This notebook gives a small end-to-end KLDM-style training example using the real code in this repo.

We will:
- load a real `DNGTask` batch from MP-20
- diffuse lattice `l` with the VP diffusion
- diffuse atom features `a` with the VP diffusion
- diffuse velocity / positions with the trivialised diffusion class
- pass the noisy variables into `CSPVNet`
- train on a simple sum of the three score-matching losses

This is still a **simple educational notebook**. It uses the current simplified `TrivialisedDiffusion` implementation from the repo, not the full final KLDM paper pipeline.

## Theory

For one crystal graph we sample a diffusion time `t` uniformly in `[eps, 1]`.

Then we build noisy training inputs:

- lattice:
  \[
  l_t = \alpha(t) l_0 + \sigma(t) \epsilon_l
  \]
- atom features:
  \[
  a_t = \alpha(t) a_0 + \sigma(t) \epsilon_a
  \]
- trivialised momentum part:
  \[
  v_t, f_t \sim p_{t|0}(f_t, v_t \mid f_0, v_0)
  \]

The network sees `(t, f_t, v_t, l_t, a_t)` and predicts three outputs:
- a velocity head `out["v"]`
- `out["l"]`
- `out["h"]`

For the velocity part we follow the KLDM training recipe more closely:

1. the network predicts an intermediate velocity quantity
2. we convert it into the actual score with the Eq. (19)-style transform
3. we compare that score to the target from the TDM forward process

For lattice and atom features we keep the same DSM form used in the continuous notebook.

In [1]:
import sys
from pathlib import Path

import torch

repo_root = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / 'src' / 'kldm').is_dir())
src_root = repo_root / 'src'
notebook_dir = repo_root / 'src' / 'kldm'
sys.path = [p for p in sys.path if Path(p or '.').resolve() != notebook_dir]
sys.path.insert(0, str(src_root))

from kldm.data import DNGTask, resolve_data_root
from kldm.distribution.uniform import sample_uniform
from kldm.kldm import ModelKLDM

torch.manual_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root = resolve_data_root()

model = ModelKLDM(device=device).to(device)

root, device, model

MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen


(PosixPath('/workspace/data'),
 device(type='cpu'),
 ModelKLDM(
   (score_network): CSPVNet(
     (act_fn): SiLU()
     (node_embedding): Linear(in_features=118, out_features=128, bias=False)
     (atom_latent_emb): Linear(in_features=256, out_features=128, bias=True)
     (dis_emb): SinEmbedding()
     (time_emb): FourierEmbedding()
     (layers): ModuleList(
       (0-3): 4 x CSPVLayer(
         (dis_emb): SinEmbedding()
         (v_proj): Linear(in_features=3, out_features=60, bias=True)
         (edge_mlp): Sequential(
           (0): Linear(in_features=382, out_features=128, bias=True)
           (1): SiLU()
           (2): Linear(in_features=128, out_features=128, bias=True)
           (3): SiLU()
         )
         (node_mlp): Sequential(
           (0): Linear(in_features=256, out_features=128, bias=True)
           (1): SiLU()
           (2): Linear(in_features=128, out_features=128, bias=True)
           (3): SiLU()
         )
         (layer_norm): LayerNorm((128,), eps=1e-

## 1. Load A Real DNG Batch

In [2]:
batch = next(
    iter(DNGTask().dataloader(root=root, split='train', batch_size=2, shuffle=False, download=True))
).to(device)

print('num graphs :', batch.num_graphs)
print('pos shape  :', tuple(batch.pos.shape))
print('h shape    :', tuple(batch.h.shape))
print('l shape    :', tuple(batch.l.shape))
print('\nfirst 3 positions:')
print(batch.pos[:3])
print('\nfirst 3 one-hot atom rows:')
print(batch.h[:3])

num graphs : 2
pos shape  : (25, 3)
h shape    : (25, 118)
l shape    : (2, 6)

first 3 positions:
tensor([[6.6651e-01, 1.7850e-03, 3.3302e-01],
        [1.2300e-04, 9.9972e-01, 2.4600e-04],
        [3.3202e-01, 9.9566e-01, 6.6405e-01]])

first 3 one-hot atom rows:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.,

## 2. Sample Noisy KLDM Inputs

In [3]:
t_graph = sample_uniform(lb=model.diffusion_l.eps, size=(batch.num_graphs, 1), device=device)
(v_t, f_t, l_t, a_t), (target_v, target_l, target_a) = model.algorithm1_training_targets(batch, t_graph)

print('t_graph shape :', tuple(t_graph.shape))
print('v_t shape     :', tuple(v_t.shape))
print('f_t shape     :', tuple(f_t.shape))
print('l_t shape     :', tuple(l_t.shape))
print('a_t shape     :', tuple(a_t.shape))
print('target_v shape:', tuple(target_v.shape))
print('target_l shape:', tuple(target_l.shape))
print('target_a shape:', tuple(target_a.shape))
print('\nfirst 3 noisy positions:')
print(f_t[:3])
print('\nfirst 3 noisy atom rows:')
print(a_t[:3])
print('\nnoisy lattice vectors:')
print(l_t)



t_graph shape: (2, 1)
l_t shape    : (2, 6)
a_t shape    : (25, 118)
f_t shape    : (25, 3)
v_t shape    : (25, 3)

first 3 noisy positions:
tensor([[0.7164, 0.0119, 0.3589],
        [0.9973, 0.9258, 0.0554],
        [0.3082, 0.0276, 0.6340]])

first 3 noisy atom rows:
tensor([[-2.0637e-01,  2.9473e-01,  1.6300e-01, -1.3779e-02, -1.6818e-01,
         -3.5226e-02,  7.7802e-02,  1.6592e-01,  4.1779e-01, -3.6882e-01,
          5.9895e-01,  3.4821e-01,  2.0365e-01, -2.6438e-01,  4.4610e-02,
         -2.9952e-01, -7.5191e-02, -2.5091e-01,  1.1544e-01, -9.8216e-02,
         -4.1297e-01,  6.0438e-02,  4.5307e-02,  5.0675e-02, -1.1522e-01,
          4.1487e-01,  3.0223e-01,  2.1139e-01,  1.1460e-01, -2.2920e-01,
          8.7176e-02,  4.8414e-02,  1.1239e-01, -2.7743e-01,  1.9033e-01,
         -3.6679e-01, -1.8590e-01,  6.9971e-03,  5.6828e-01, -4.8867e-02,
          1.8715e-01, -2.6179e-01,  3.8796e-01, -5.9039e-02, -2.2214e-02,
         -5.3740e-01,  1.3573e-01,  3.5114e-01,  5.0072e-01,  3.

## 3. Score Network training

In [ ]:
def tdm_paper_lambda(t_node: torch.Tensor) -> torch.Tensor:
    """Velocity weighting used by the current KLDM training path."""
    return torch.full_like(t_node, model.tdm.time_scaling_T ** 2)


optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model



ModelKLDM(
  (score_network): CSPVNet(
    (act_fn): SiLU()
    (node_embedding): Linear(in_features=118, out_features=128, bias=False)
    (atom_latent_emb): Linear(in_features=256, out_features=128, bias=True)
    (dis_emb): SinEmbedding()
    (time_emb): FourierEmbedding()
    (layers): ModuleList(
      (0-3): 4 x CSPVLayer(
        (dis_emb): SinEmbedding()
        (v_proj): Linear(in_features=3, out_features=60, bias=True)
        (edge_mlp): Sequential(
          (0): Linear(in_features=382, out_features=128, bias=True)
          (1): SiLU()
          (2): Linear(in_features=128, out_features=128, bias=True)
          (3): SiLU()
        )
        (node_mlp): Sequential(
          (0): Linear(in_features=256, out_features=128, bias=True)
          (1): SiLU()
          (2): Linear(in_features=128, out_features=128, bias=True)
          (3): SiLU()
        )
        (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
    )
    (final_layer_norm): LayerNorm

## 5. Simple Training Loop

This is the same training pattern as the other notebooks: sample a batch, sample random times, build noisy inputs from the forward kernels, predict the scores, and regress against the exact targets.

In [ ]:
train_loader = DNGTask().dataloader(root=root, split='train', batch_size=64, shuffle=True, download=True)
num_epochs = 5
print_every = 20

for epoch in range(num_epochs):
    running = {'loss': 0.0, 'loss_v': 0.0, 'loss_l': 0.0, 'loss_a': 0.0}

    for step, batch in enumerate(train_loader):
        batch = batch.to(device)
        t_graph = sample_uniform(lb=model.diffusion_l.eps, size=(batch.num_graphs, 1), device=device)

        optimizer.zero_grad()
        loss, metrics = model.algorithm2_loss(
            batch=batch,
            t=t_graph,
            lambda_v=1.0,
            lambda_l=1.0,
            lambda_a=1.0,
            lambda_t_fn=tdm_paper_lambda,
        )
        loss.backward()
        optimizer.step()

        for key in running:
            if key in metrics:
                running[key] += float(metrics[key])

        if step % print_every == 0:
            print(
                f'epoch={epoch:02d} step={step:03d} '
                f'loss={float(metrics["loss"]):.6f} '
                f'(v={float(metrics["loss_v"]):.4f}, '
                f'l={float(metrics["loss_l"]):.4f}, '
                f'a={float(metrics["loss_a"]):.4f})'
            )

    n_steps = step + 1
    for key in running:
        running[key] /= n_steps

    print(
        f'epoch={epoch:02d} mean_loss={running["loss"]:.6f} '
        f'(v={running["loss_v"]:.4f}, l={running["loss_l"]:.4f}, a={running["loss_a"]:.4f})'
    )



epoch=00 step=000 loss=1.375817 (v=1.2103, l=0.1111, a=0.0545)
epoch=00 step=020 loss=1.490954 (v=1.2883, l=0.1442, a=0.0584)
epoch=00 step=040 loss=1.169565 (v=1.0091, l=0.1114, a=0.0491)
epoch=00 step=060 loss=1.718090 (v=1.5743, l=0.0949, a=0.0488)
epoch=00 step=080 loss=1.676626 (v=1.5377, l=0.0935, a=0.0454)
epoch=00 step=100 loss=2.097816 (v=1.9216, l=0.1287, a=0.0474)
epoch=00 step=120 loss=1.265485 (v=1.1000, l=0.1237, a=0.0418)
epoch=00 step=140 loss=1.391617 (v=1.2296, l=0.1162, a=0.0458)
epoch=00 step=160 loss=1.705662 (v=1.5324, l=0.1248, a=0.0484)
epoch=00 step=180 loss=1.265854 (v=1.0541, l=0.1741, a=0.0377)
epoch=00 step=200 loss=1.162188 (v=1.0295, l=0.0967, a=0.0360)
epoch=00 step=220 loss=1.738820 (v=1.5422, l=0.1546, a=0.0420)
epoch=00 step=240 loss=1.934209 (v=1.7392, l=0.1390, a=0.0560)
epoch=00 step=260 loss=1.399842 (v=1.2683, l=0.0981, a=0.0334)
epoch=00 step=280 loss=1.408084 (v=1.2824, l=0.0917, a=0.0339)
epoch=00 step=300 loss=1.043386 (v=0.9348, l=0.0744, a=